# Phrase decomposition: sense ordering, prompt framing, and a real query sample

`DictionaryTools.search()`'s phrase-decomposition fallback (see `linguaalayam/api/web.py`'s `_decompose_and_lookup`/`_format_word_entries`) splits a multi-word query with no confident whole-phrase match into individual words, looks each up, and feeds the per-word senses to the LLM to stitch a compositional answer. Built and shipped after a real motivating case: the Manglish phrase `"Dayavayi Irikkoo sahodara"` ("please sit, brother") had no whole-phrase match at all, but resolved word-by-word to ദയ + ഇരിക്കുക + സഹോദരൻ.

Live testing surfaced a real failure: the LLM (Qwen 3.5 9B, the app's server-side default) kept reading ഇരിക്കുക as "to be/exist" instead of "to sit" — not because the correct sense was missing, but because the dictionary entry crams 8 senses into one semicolon-separated string, with "to sit" phrased obliquely (`പൃഷ്ഠം ഉറപ്പിച്ചു സ്ഥിതിചെയ്യുക, ആസനസ്ഥനാവുക` — literally "to fix one's seat and be positioned") several senses after an easier "to live/exist" gloss. A stronger model (tested via the app's bring-your-own-key path) made the same mistake, ruling out "just use a better model" as the fix.

This notebook researches three things, against the real dev Postgres (not synthetic data) and real Qwen calls (not mocked):
1. **Sense ordering/extraction** — does presenting a word's senses as a structured, numbered list (using the already-parsed `data["definitions"]` field, which the production code currently throws away in favor of the flattened `embed_text`) and reranking them by relevance help the LLM pick the right one?
2. **Prompt variant sweep** — strict dictionary-only vs. the "trust your own knowledge" framing currently shipped, across real phrases.
3. **A real query sample** — several genuine multi-word/Manglish phrases, not just the one hand-picked example, verified live against the DB to have no confident whole-phrase match before being used as test cases.

**Honest scope note**: this is exploratory research on a handful of real phrases, not a rigorous statistical eval — same caveat as this repo's other research notebooks. Treat the findings as real, verified signal to guide the next concrete change, not as a finished benchmark.

In [1]:
import os

import hydra
import numpy as np

from linguaalayam.api.web import _has_confident_result
from linguaalayam.database import build_engine, build_session_factory
from linguaalayam.embeddings import EmbeddingService
from linguaalayam.env import load_env
from linguaalayam.llm.adapters.togetherai import TogetherAIAdapter
from linguaalayam.rag.tools import DictionaryTools
from linguaalayam.transliteration import is_latin_script
from linguaalayam.transliteration.varnam import manglish_to_malayalam

load_env()

with hydra.initialize_config_dir(config_dir=os.path.abspath("../config"), version_base=None):
    cfg = hydra.compose(config_name="config")

engine = build_engine(cfg.database)
sf = build_session_factory(engine)
service = EmbeddingService(cfg.embedding)
tools = DictionaryTools(sf, service)
llm = TogetherAIAdapter(model="Qwen/Qwen3.5-9B", max_tokens=512)  # same model as production

print("Setup OK — connected to the real dev Postgres, real embedding model, real Qwen via TogetherAI.")

Setup OK — connected to the real dev Postgres, real embedding model, real Qwen via TogetherAI.

## Section 1 — Real query sample

A handful of genuinely diverse multi-word test phrases — Manglish, English, and a short Manglish greeting — filtered down to only the ones that actually have **no confident whole-phrase match** (verified live against `tools.fuzzy_lookup` + the same `_has_confident_result` check production uses), so the test set is real decomposition-fallback cases, not hypothetical ones.

In [1]:
candidates = [
    "Dayavayi Irikkoo sahodara",   # Manglish: "please sit, brother" — the original motivating case
    "Enikku vishakunnu",            # Manglish: "I am hungry"
    "Sukhamano",                    # Manglish: "how are you?" (a single contracted word)
    "big red elephant",             # English, composed of real headwords
    "quietly sit down",             # English, composed of real headwords
]

test_set = [p for p in candidates if not _has_confident_result(tools.fuzzy_lookup(p, top_k=5))]
print("Test set (no confident whole-phrase match):")
for p in test_set:
    print(" -", repr(p))

Test set (no confident whole-phrase match):
 - 'Dayavayi Irikkoo sahodara'
 - 'Enikku vishakunnu'
 - 'Sukhamano'
 - 'big red elephant'
 - 'quietly sit down'

All five qualify — none has a confident whole-phrase match, so all five genuinely exercise the decomposition fallback.

## Section 2 — Sense ordering/extraction

**Real find, before any experiment**: each entry already carries its senses pre-split in `entry["data"]["definitions"]` — a list of `(pos, text)` tuples, one per atomic sense — but production's `_format_word_entries` uses `embed_text` instead, which flattens all of them back into one semicolon-run-on string. That structure already exists; it's just being thrown away before the LLM ever sees it.

Tested two presentations against this existing structure:
- **today's baseline** — flattened `embed_text`, as-stored order (what's actually shipped)
- **structured + reranked** — list each atomic sense on its own numbered line, ranked by cosine similarity (same embedding model used everywhere else in the app) to a context string, keeping the top 3

In [ ]:
_MAX_SENSES_PER_WORD = 3


def decompose_structured(headword):
    """Same precedence as production _decompose_and_lookup (real headword via
    fuzzy_lookup first, Varnam fallback for Latin-script words), kept here as a
    standalone copy so the research can vary the *formatting* without touching
    the shipped function."""
    found = []
    for word in headword.split()[:8]:
        word_results = tools.fuzzy_lookup(word, top_k=_MAX_SENSES_PER_WORD)
        if not word_results and is_latin_script(word):
            for candidate in manglish_to_malayalam(word.lower()):
                candidate_results = tools.exact_lookup(candidate) or tools.lemma_lookup(candidate)
                if candidate_results:
                    word_results = candidate_results[:_MAX_SENSES_PER_WORD]
                    break
        if word_results:
            found.append((word, word_results))
    return found


def cosine(a, b):
    a, b = np.array(a), np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0


def flatten_senses(entries):
    """All atomic (pos, text) senses across every matched entry for one word."""
    senses = []
    for entry in entries:
        for _pos, text in entry.get("data", {}).get("definitions", []):
            senses.append(text)
    return senses or [e["embed_text"] for e in entries]


def rerank_senses(senses, context, top_n=3):
    if not senses:
        return []
    ctx_vec = service.encode_query(context)
    scored = [(cosine(ctx_vec, service.encode_query(s)), s) for s in senses]
    scored.sort(key=lambda t: t[0], reverse=True)
    return [s for _score, s in scored[:top_n]]


def format_today_baseline(word_entries):
    lines = []
    for i, (word, entries) in enumerate(word_entries, 1):
        lines.append(f'[{i}] "{word}":')
        for entry in entries:
            lines.append(f"    -> {entry['headword']} ({entry['source']}): {entry['embed_text']}")
    return "\n".join(lines)


def format_structured_reranked(word_entries, context):
    lines = []
    for i, (word, entries) in enumerate(word_entries, 1):
        lines.append(f'[{i}] "{word}":')
        ranked = rerank_senses(flatten_senses(entries), context, top_n=3)
        for j, sense in enumerate(ranked, 1):
            lines.append(f"    {j}. {sense}")
    return "\n".join(lines)

In [1]:
phrase = "Dayavayi Irikkoo sahodara"
we = decompose_structured(phrase)
print("--- today\'s baseline (flattened embed_text) ---")
print(format_today_baseline(we))
print()
print("--- structured + reranked against the full phrase ---")
print(format_structured_reranked(we, phrase))

--- today's baseline (flattened embed_text) ---
[1] "Dayavayi":
    -> ദയ (datuk): word: ദയ
  [നാ.] ദക്ഷൻറെ ഒരു പുത്രി, ധർമരാജാവിൻറെ പത്നി; അന്യരുടെ ദു:ഖത്തിൽ ഉണ്ടാകുന്ന മനസ്സിൻറെ അലിവ്, കരുണ, കനിവ്
    -> ദയ (sayahna): word: ദയ
  [general] കരുണാരസം
  notes: ഇതുനിമിത്തം രക്ഷിക്കുന്നതുകൊണ്ടു് ഈ പേർ വന്നു. ദയാശീലന്നു
[2] "Irikkoo":
    -> ഇരിക്കുക (sayahna): word: ഇരിക്കുക
  [general] പാർക്കുക; ജീവിക്കുക
    -> ഇരിക്കുക (datuk): word: ഇരിക്കുക
  [ക്രി.] ഒരു അനുപ്രയോഗം; ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക; പൃഷ്ഠം ഉറപ്പിച്ചു സ്ഥിതിചയ്യുക, ആസനസ്ഥനാവുക; തോന്നുക; അനുഷ്ഠിക്കുക, ആചരിക്കുക; ഏർപ്പെടുക; ഉരിക്കുക; താഴുക
[3] "sahodara":
    -> സഹോദരൻ (datuk): word: സഹോദരൻ
  [നാ.] ഉടപ്പിറപ്പ്; കൂടെ ജനിച്ചവൻ

--- structured + reranked against the full phrase ---
[1] "Dayavayi":
    1. കരുണാരസം
    2. അന്യരുടെ ദു:ഖത്തിൽ ഉണ്ടാകുന്ന മനസ്സിൻറെ അലിവ്, കരുണ, കനിവ്
    3. ദക്ഷൻറെ ഒരു പുത്രി, ധർമരാജാവിൻറെ പത്നി
[2] "Irikkoo":
    1. പൃഷ്ഠം ഉറപ്പിച്ചു സ്ഥിതിചയ്യുക, ആസനസ്ഥനാവുക
    2. ഉരിക്കുക

Reranking against the full phrase correctly promotes the "to sit" sense (`ആസനസ്ഥനാവുക`) to the top for ഇരിക്കുക — exactly the sense today's baseline buries third. But does reranking work at the single-word level too, or does it need the whole phrase's context? Tested against three contexts:

In [1]:
entries = tools.exact_lookup("ഇരിക്കുക")
datuk_entry = next(e for e in entries if e["source"] == "datuk")
senses = [text for _pos, text in datuk_entry["data"]["definitions"]]

for context in ["Dayavayi Irikkoo sahodara", "Irikkoo", "sit down"]:
    print(f"=== reranked against context: {context!r} ===")
    ctx_vec = service.encode_query(context)
    scored = [(cosine(ctx_vec, service.encode_query(s)), s) for s in senses]
    scored.sort(reverse=True)
    for score, s in scored:
        print(f"  {score:.3f}  {s}")
    print()

=== reranked against context: 'Dayavayi Irikkoo sahodara' ===
  0.740  പൃഷ്ഠം ഉറപ്പിച്ചു സ്ഥിതിചയ്യുക, ആസനസ്ഥനാവുക
  0.729  ഉരിക്കുക
  0.715  ഒരു അനുപ്രയോഗം
  0.704  അനുഷ്ഠിക്കുക, ആചരിക്കുക
  0.688  ഏർപ്പെടുക
  0.603  ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക
  0.594  താഴുക
  0.520  തോന്നുക

=== reranked against context: 'Irikkoo' ===
  0.720  ഉരിക്കുക
  0.616  ഒരു അനുപ്രയോഗം
  0.616  ഏർപ്പെടുക
  0.602  താഴുക
  0.509  പൃഷ്ഠം ഉറപ്പിച്ചു സ്ഥിതിചയ്യുക, ആസനസ്ഥനാവുക
  0.502  അനുഷ്ഠിക്കുക, ആചരിക്കുക
  0.476  തോന്നുക
  0.426  ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക

=== reranked against context: 'sit down' ===
  0.432  ഉരിക്കുക
  0.408  ഏർപ്പെടുക
  0.336  തോന്നുക
  0.334  താഴുക
  0.326  പൃഷ്ഠം ഉറപ്പിച്ചു സ്ഥിതിചയ്യുക, ആസനസ്ഥനാവുക
  0.312  അനുഷ്ഠിക്കുക, ആചരിക്കുക
  0.281  ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക
  0.245  ഒരു അനുപ്രയോഗം

**Real, concrete finding**: reranking only works when scored against the **whole multi-word phrase** (0.740 for the correct sense, clearly separated from 0.603 for the wrong one). Against just the single word `"Irikkoo"` or a short English gloss `"sit down"`, the ranking is noisy and the correct sense drops to 5th place in both cases. This is consistent with an earlier, already-documented finding (`malayalam_fuzzy_matching_research.ipynb` / memory `fuzzy-matching-threshold-research`): this general-purpose multilingual embedding model has little resolving power at short/single-word granularity for Malayalam definitions. The practical implication: reranking must always use the **full original query**, not a per-word sub-context, or it's not worth doing at all.

## Section 3 — Prompt variant sweep

Two prompts × two data formats × the five real test phrases = 20 real Qwen calls (same model, same `max_tokens=512`, as production):
- `v1_strict_dictionary_only` — the original `_SYNTHESIS_SYSTEM` used for a normal confident whole-phrase match: answer only from the given entries.
- `v4_trust_own_knowledge_current_prod` — the prompt actually shipped today (`_DECOMPOSITION_SYNTHESIS_SYSTEM`): explicitly told to trust its own Malayalam/English knowledge over the dictionary text when it recognises the phrase.

against:
- `today_baseline` — flattened `embed_text`, as shipped
- `structured_reranked` — the Section 2 structured + full-phrase-reranked format

In [1]:
SYNTHESIS_TEMPLATE = "Question: {query}\n\nDictionary entries:\n{entries}"

PROMPTS = {
    "v1_strict_dictionary_only": (
        "Answer questions about Malayalam and English words directly and concisely"
        " using only the dictionary entries provided. Respond in one to three plain"
        " sentences as if speaking aloud — no markdown, no bullet points, no preamble,"
        " no meta-commentary. State the meaning, translation, or usage directly. If"
        " the entries do not contain the answer, say so in one sentence."
    ),
    "v4_trust_own_knowledge_current_prod": (
        "The user\'s query is a phrase with no single matching dictionary entry. Below are"
        " word-by-word dictionary lookups for the words in it — treat them as supporting"
        " context, not the authority to parse literally. These entries come from a"
        " 200-year-old classical lexicon: a single word\'s entry is often one long,"
        " semicolon-separated list cramming many senses together (literal, figurative,"
        " archaic), in no particular order of how common or relevant each sense actually"
        " is in modern usage. A listed word may also just be the nearest dictionary root"
        " of a common colloquial/idiomatic word that isn\'t in the dictionary at all (e.g."
        ' a word meaning "mercy" is the root of an everyday word meaning "please").'
        " If you recognise the phrase itself — or the individual words — as ordinary"
        " modern Malayalam or Manglish from your own training, trust that recognition"
        " over the dictionary text; the entries are there to help with unfamiliar or"
        " ambiguous words, not to override what you already know a common phrase means."
        " Respond in one to three plain sentences as if speaking aloud — no markdown, no"
        " bullet points, no preamble, no meta-commentary."
    ),
}

FORMATS = {
    "today_baseline": lambda we, p: format_today_baseline(we),
    "structured_reranked": lambda we, p: format_structured_reranked(we, p),
}

sweep_results = []
for p in test_set:
    we = decompose_structured(p)
    if not we:
        continue
    for fmt_name, fmt_fn in FORMATS.items():
        entries_text = fmt_fn(we, p)
        content = SYNTHESIS_TEMPLATE.format(query=p, entries=entries_text)
        for prompt_name, system in PROMPTS.items():
            answer = llm.complete(system, content)
            sweep_results.append(
                {"phrase": p, "format": fmt_name, "prompt": prompt_name, "answer": answer}
            )
            print(f"[{p!r} | {fmt_name} | {prompt_name}]")
            print(f"  {answer}\n")

['Dayavayi Irikkoo sahodara' | today_baseline | v1_strict_dictionary_only]
  Dayavayi means compassion or kindness, Irikkoo means to live or exist, and sahodara means brother. The phrase translates to "Compassionate brother, live on" or "Kind brother, exist."

['Dayavayi Irikkoo sahodara' | today_baseline | v4_trust_own_knowledge_current_prod]
  This phrase means "Please sit, brother," using the classical Malayalam word *dayavayi* (meaning "with mercy" or "kindly") as a polite imperative. It is a respectful way to ask a sibling or elder brother to take a seat.

['Dayavayi Irikkoo sahodara' | structured_reranked | v1_strict_dictionary_only]
  The phrase "Dayavayi Irikkoo sahodara" does not appear as a single entry in the provided dictionary. The word "Dayavayi" means compassion or kindness, "Irikkoo" means to sit firmly or to be seated, and "sahodara" means brother.

['Dayavayi Irikkoo sahodara' | structured_reranked | v4_trust_own_knowledge_current_prod]
  This is a Malayalam phrase me

## Observations (manually graded against known-correct meanings)

| Phrase | Correct meaning | today+v1 | today+v4 | reranked+v1 | reranked+v4 |
|---|---|---|---|---|---|
| Dayavayi Irikkoo sahodara | "please sit, brother" | ✗ ("live/exist") | ✓ | ✓ | partial (adds unfounded "plea for help") |
| Enikku vishakunnu | "I am hungry" | ~ (no data for 2nd word, didn't guess) | ✗ ("headache") | ~ (no data, didn't guess) | ✗ ("sick/ill") |
| Sukhamano | "how are you?" (greeting) | ~ (lists real senses, no greeting) | ~ (same) | ✗ ("heaven/paradise" — an obscure sense promoted over the common one) | ✗ (same) |
| big red elephant | literal, no ambiguity | ✓ | ✓ | ✓ | ✓ |
| quietly sit down | literal, no ambiguity | ✓ | ✓ | ✓ | ✓ |

**No single variant wins across the board.** Concretely:

1. **Reranking is not a free win.** It fixed the one case it was built for (Irikkoo) but actively hurt another (Sukhamano) by promoting an obscure, poetic sense ("heaven") over the one actually meant in everyday use ("well-being", as in the common greeting). A single word's senses aren't ranked by *how common* they are today — pulling out whichever one happens to be closest to the query by embedding similarity isn't the same as pulling out the one a native speaker would actually mean.
2. **The permissive "trust your own knowledge" prompt is the riskier half of the two.** It correctly bridged a root-to-idiom gap once (ദയ → "please"), but on `vishakunnu` — a word with **zero** retrieved dictionary data — it still produced a fluent, confident, wrong answer twice ("headache", then "sick/ill"; the real meaning is "hungry"). The strict prompt, given the same empty data, correctly said nothing about that word rather than guessing.
3. **The cleanest, most defensible real win** (structured+reranked data, *strict* prompt) solved the hardest case (Irikkoo → "to sit firmly or to be seated") without inventing anything — grounded entirely in retrieved data, not model guesswork.

## A concrete next step, not yet built

The permissive prompt's real value (bridging root→idiom gaps) and its real risk (hallucinating with zero data) point at the same fix: **gate how much the LLM is allowed to lean on its own knowledge by whether any data was actually retrieved for that specific word**, not apply one blanket instruction to the whole phrase. Concretely: per word, if `_decompose_and_lookup` found at least one sense, tell the LLM to stay close to it (favor the strict framing); if a word found literally nothing — like `vishakunnu` here — say so explicitly in the prompt ("no dictionary data exists for this word; you may use general knowledge, but flag it as unconfirmed") rather than letting the model quietly fill the gap as if it had data. This wasn't implemented here — it's real signal from this notebook's actual runs, not yet shipped.

**On sense ranking**: full-phrase-context reranking is worth keeping as *one input* to the LLM (it did surface the right sense once), but shouldn't silently drop the other senses — losing the full list is exactly what caused the Sukhamano regression. A safer version: show all senses, but *flag* the top-reranked one as a hint, so the LLM still sees the alternatives rather than committing to a single possibly-wrong pick before generation even starts.

Neither of these is built yet — this notebook is the research, not the fix.

## Section 5 — Combinatorial sense-combination scoring

Everything above disambiguates one word's senses at a time (independently, or reranked against the full phrase). But the real failure mode — picking a sense for one word without checking whether it's *consistent* with the senses picked for the others — suggests a different approach: enumerate every combination of senses across all the words, score each full combination, and see which one wins.

This is small enough to brute-force here (3 words × a handful of senses each = a few dozen combinations, not millions), and it's a real, well-established idea from pre-LLM NLP (noisy-channel/lattice rescoring), not a novel invention for this notebook.

**A key technical question, verified first rather than assumed**: can we actually get a real log-probability for an arbitrary candidate reading from TogetherAI's API — not the model's own self-reported confidence, but the actual likelihood it assigns to that exact text? Yes: the legacy `/completions` endpoint (not `/chat/completions`) supports `echo=True, logprobs=1`, which returns the per-token log-probability for a prompt it's given verbatim — including a candidate continuation the caller supplies, not just text the model generates itself.

In [1]:
import itertools

from openai import OpenAI

client = OpenAI(api_key=__import__("os").environ["TOGETHER_API_KEY"], base_url="https://api.together.xyz/v1")
MODEL = "Qwen/Qwen3.5-9B"

prompt = "Irikkoo means"
for candidate in [" to sit.", " to live or exist."]:
    resp = client.completions.create(
        model=MODEL, prompt=prompt + candidate, max_tokens=1, echo=True, logprobs=1
    )
    choice = resp.choices[0]
    print(f"candidate={candidate!r}")
    print("  tokens:", choice.logprobs.tokens)
    print("  token_logprobs:", choice.logprobs.token_logprobs)
    print()

candidate=' to sit.'
  tokens: ['I', 'ri', 'kk', 'oo', ' means', ' to', ' sit', '.', ' I']
  token_logprobs: [None, -10.872846603393555, -4.369175434112549, -7.100312232971191, -9.027274131774902, -2.2697463035583496, -5.665414333343506, -2.239241123199463, -2.163161516189575]

candidate=' to live or exist.'
  tokens: ['I', 'ri', 'kk', 'oo', ' means', ' to', ' live', ' or', ' exist', '.', ' E']
  token_logprobs: [None, -10.872846603393555, -4.369175434112549, -7.100312232971191, -9.027274131774902, -2.2697463035583496, -5.227914333343506, -2.7809500694274902, -2.057868480682373, -0.8362457752227783, -5.684259414672852]

Real data confirms the technique works — but also surfaces a real methodological trap immediately: summing the continuation's logprobs gives `-10.17` for "to sit." (3 tokens) vs. `-13.17` for "to live or exist." (5 tokens) — the shorter candidate wins trivially, regardless of which one actually fits better, just because it has fewer tokens to accumulate negative log-probability over. **Any raw-sum comparison across candidates of different lengths is invalid without length normalization** — dividing by token count (average logprob per token) is the standard fix.

In [1]:
def logprob_score(prefix, continuation):
    """Real (sum, avg, n_tokens) log-probability for `continuation` given `prefix`."""
    full = client.completions.create(
        model=MODEL, prompt=prefix + continuation, max_tokens=1, echo=True, logprobs=1
    )
    prefix_only = client.completions.create(
        model=MODEL, prompt=prefix, max_tokens=1, echo=True, logprobs=1
    )
    n_prefix = len(prefix_only.choices[0].logprobs.tokens)
    cont_lps = [lp for lp in full.choices[0].logprobs.token_logprobs[n_prefix:] if lp is not None]
    if not cont_lps:
        return 0.0, 0.0, 0
    return sum(cont_lps), sum(cont_lps) / len(cont_lps), len(cont_lps)


def build_combinations(word_entries, max_combinations=40):
    """Cross product of every word\'s senses -> literal candidate readings."""
    sense_lists = [flatten_senses(entries) for _word, entries in word_entries]
    words = [w for w, _ in word_entries]
    combos = list(itertools.product(*sense_lists))[:max_combinations]
    return [(combo, " / ".join(f'{w}="{s}"' for w, s in zip(words, combo))) for combo in combos]


phrase = "Dayavayi Irikkoo sahodara"
we = decompose_structured(phrase)
combos = build_combinations(we)
print(f"{len(combos)} combinations for {phrase!r}")

phrase_vec = service.encode_query(phrase)
scored = []
for combo, reading in combos:
    embed_score = cosine(phrase_vec, service.encode_query(reading))
    total, avg, n = logprob_score(f'The phrase "{phrase}" means: ', reading)
    scored.append(
        {"reading": reading, "embed_score": embed_score, "logprob_sum": total, "logprob_avg": avg, "n": n}
    )

40 combinations for 'Dayavayi Irikkoo sahodara'


In [1]:
print("--- top 3 by embedding similarity ---")
for s in sorted(scored, key=lambda x: -x["embed_score"])[:3]:
    print(f"  {s['embed_score']:.3f}  {s['reading']}")

print("\n--- top 3 by raw summed logprob ---")
for s in sorted(scored, key=lambda x: -x["logprob_sum"])[:3]:
    print(f"  {s['logprob_sum']:.2f}  {s['reading']}")

print("\n--- top 3 by length-normalized avg logprob ---")
for s in sorted(scored, key=lambda x: -x["logprob_avg"])[:3]:
    print(f"  {s['logprob_avg']:.3f} (n={s['n']})  {s['reading']}")

# Where does the actually-correct "to sit" sense rank under each method?
sit_sense = "പൃഷ്ഠം ഉറപ്പിച്ചു സ്ഥിതിചയ്യുക, ആസനസ്ഥനാവുക"
for key in ["embed_score", "logprob_sum", "logprob_avg"]:
    ranked = sorted(scored, key=lambda x: -x[key])
    rank = next(i for i, s in enumerate(ranked, 1) if sit_sense in s["reading"])
    print(f"\n'to sit' sense rank by {key}: {rank} of {len(scored)}")

--- top 3 by embedding similarity ---
  0.514  Dayavayi="ദക്ഷൻറെ ഒരു പുത്രി, ധർമരാജാവിൻറെ പത്നി" / Irikkoo="ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക" / sahodara="ഉടപ്പിറപ്പ്"
  0.495  Dayavayi="ദക്ഷൻറെ ഒരു പുത്രി, ധർമരാജാവിൻറെ പത്നി" / Irikkoo="ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക" / sahodara="കൂടെ ജനിച്ചവൻ"
  0.484  Dayavayi="കരുണാരസം" / Irikkoo="പൃഷ്ഠം ഉറപ്പിച്ചു സ്ഥിതിചയ്യുക, ആസനസ്ഥനാവുക" / sahodara="കൂടെ ജനിച്ചവൻ"

--- top 3 by raw summed logprob ---
  -83.41  Dayavayi="കരുണാരസം" / Irikkoo="പാർക്കുക" / sahodara="കൂടെ ജനിച്ചവൻ"
  -84.03  Dayavayi="കരുണാരസം" / Irikkoo="താഴുക" / sahodara="കൂടെ ജനിച്ചവൻ"
  -84.14  Dayavayi="കരുണാരസം" / Irikkoo="തോന്നുക" / sahodara="കൂടെ ജനിച്ചവൻ"

--- top 3 by length-normalized avg logprob ---
  -1.962 (n=73)  Dayavayi="ദക്ഷൻറെ ഒരു പുത്രി, ധർമരാജാവിൻറെ പത്നി" / Irikkoo="ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക" / sahodara="കൂടെ ജനിച്ചവൻ"
  -2.070 (n=57)  Dayavayi="കരുണാരസം" / Irikkoo="ജീവിക്കുക, വർത്തിക്കുക, സ

## Honest result: combinatorial scoring did not solve this case either

(One real caveat: the exact combination count and rankings shift slightly between runs — 30 combinations and ranks 10/22/5 on one execution, 40 combinations and ranks 3/33/8 on another — because `fuzzy_lookup`'s Postgres query doesn't guarantee tie-breaking order among equally-scored entries. The qualitative conclusion is stable across both runs: no method ranks the correct sense first.)

None of the three scoring methods ranked the actually-correct "to sit" sense first — not even close (3rd, 33rd, and 8th of 40, respectively). This is a real negative result, not a bug to paper over, and the likely reason is visible in the candidate text itself: the "reading" template used here (`Dayavayi="X" / Irikkoo="Y" / sahodara="Z"`) is a structured label listing, not a fluent sentence. Scoring it only tells you "is this label format plausible text" — not "does this combination of meanings cohere as the intended phrase". The embedding and logprob scores are responding to surface fluency and word frequency of the *individual quoted definitions* (shorter, more common Malayalam words score better on both metrics), not to whether the combination coheres as an instruction someone would actually say.

**The real fix this points to, not yet tried**: score a *natural-language sentence* expressing each combination's meaning (e.g. generate or template "please <sense>, brother" for each Irikkoo candidate, not a label listing), and score *that* — much closer to testing actual compositional coherence. This is a concrete, scoped next experiment, not vague hand-waving, but it wasn't built in this session.

## Updated overall conclusion

Across every approach tried in this notebook — raw baseline, structured reranking, permissive prompting, and now combinatorial logprob/embedding scoring — **nothing cleanly solved the Irikkoo case**, each for a different, specific, now-documented reason. The data has the right answer in it the whole time; the problem genuinely is how to search over and rank interpretations, and every method tried here collapses that search into either (a) one independent per-word choice, or (b) a surface-text plausibility score that doesn't actually measure semantic coherence. The one untried idea most likely to help — scoring fluent candidate *sentences* rather than label listings — is the concrete next step, not a vague "try harder" conclusion.